# Sentinel-X — Person detection (YOLO) + face landmarks

Run the cells **top to bottom**. The pipeline has two stages:

1. **YOLO** finds every *person* in the full-resolution frame (a body stays visible much farther away than a face) and tracks them with an ID.
2. **MediaPipe Face Landmarker** runs only on the *head area* of each person, zoomed in, so far-away faces are still found.

| Part | What it does |
|------|--------------|
| 1 | Install the libraries (once) |
| 2 | **All settings in one place** |
| 3 | Imports + download the face model |
| 4 | Small helpers (drawing, geometry) |
| 5 | The detection pipeline (YOLO persons → face on each head) |
| 6 | The camera loop (only *defines* it, does not open the camera) |
| 7 | **Run**: loads and warms up everything, *then* opens the camera, once |

Keys in the video window: **q** or **X** = quit, **b** = person boxes on/off, **m** = face mesh, **l** = landmark dots.
Nothing is saved.

## Part 1 — Install (once)

`ultralytics` provides YOLO (it installs PyTorch as a dependency, so it is a big download).
`opencv-python` reads the webcam and draws, `mediapipe` provides the face model.

> If the install fails on a very new Python version (PyTorch wheels sometimes lag behind), create a virtual environment with Python 3.12 or 3.13 and run the notebook there.

In [11]:
%pip install -U ultralytics opencv-python mediapipe
%pip install -U ultralytics

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: C:\Users\grym\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: C:\Users\grym\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


## Part 2 — Settings

Everything you may want to tune is here. If you change something, re-run **Parts 2 → 7**.

**Camera / display**
- `CAM_W` / `CAM_H`: resolution requested from the camera. More pixels = people detected farther away.
- `WIDTH`: width of the displayed window (display only, detection uses the full resolution).

**YOLO (persons)**
- `MODEL_NAME`: model size. `n` = fastest, `s` = balanced, `m` / `l` = better at long range but need a GPU to stay smooth.
- `YOLO_IMGSZ`: image size YOLO works at. Bigger = smaller/farther people found, slower. Must be a multiple of 32.
- `PERSON_CONF`: lower = more sensitive (and more false positives).

**Face (inside each person's head area)**
- `HEAD_FRAC`: top part of the person box that is searched for a face.
- `FACES_PER_FRAME` / `FACE_EVERY`: how many people get a face scan per frame, and the minimum number of frames between two scans of the same person. Between scans the face follows the person's box, so the display stays smooth.

In [27]:
# --- camera / display ---
SOURCE = 0                      # 0 = default webcam, 1 = second camera, or a video file path
CAM_W, CAM_H = 1920, 1080       # requested camera resolution (try 3840, 2160 for 4K)
WIDTH  = 1280                   # width of the displayed window
WINDOW = "Sentinel-X camera (q or X to quit)"

# --- YOLO person detector ---
MODEL_NAME     = "yolo26s.pt"   # yolo26n / s / m / l  (downloaded automatically the first time)
MODEL_FALLBACK = "yolo11s.pt"   # used if MODEL_NAME cannot be loaded (e.g. older ultralytics)
YOLO_IMGSZ     = 1280           # bigger = sees smaller (farther) people, slower. Multiple of 32.
PERSON_CONF    = 0.25           # person confidence threshold
TRACKER        = "bytetrack.yaml"
DEVICE         = None           # None = automatic (GPU if available), or "cpu", "0", "mps"

# --- face landmarks on each head ---
FACE_CONF       = 0.3           # face confidence threshold
HEAD_FRAC       = 0.4           # top 40 % of the person box is searched for a face
FACE_MIN_SIDE   = 256           # small head crops are enlarged to at least this many px
FACES_PER_FRAME = 3             # max face scans per frame
FACE_EVERY      = 2             # min frames between two face scans of the same person

## Part 3 — Imports and face model

The face model file (~4 MB) is downloaded **once** next to the notebook.
The YOLO weights are downloaded automatically by `ultralytics` the first time they are loaded (Part 7).

In [28]:
import os
import time
import urllib.request

import cv2
import numpy as np
import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision
from ultralytics import YOLO

FACE_MODEL_PATH = "face_landmarker.task"
FACE_MODEL_URL  = ("https://storage.googleapis.com/mediapipe-models/face_landmarker/"
                   "face_landmarker/float16/1/face_landmarker.task")

if not os.path.exists(FACE_MODEL_PATH):
    print("Downloading face model...")
    urllib.request.urlretrieve(FACE_MODEL_URL, FACE_MODEL_PATH)
print("Face model ready:", FACE_MODEL_PATH)

Face model ready: face_landmarker.task


## Part 4 — Helpers

- `draw_connections`: draws lines between landmarks (face contour, irises, mesh).
- `bbox`: bounding box of a set of points.
- `describe_device`: tells you whether YOLO will run on a GPU or the CPU.

In [29]:
Conn = vision.FaceLandmarksConnections


def draw_connections(frame, pts, connections, color, thickness=1):
    for c in connections:
        cv2.line(frame, pts[c.start], pts[c.end], color, thickness, cv2.LINE_AA)


def bbox(pts):
    xs, ys = zip(*pts)
    return min(xs), min(ys), max(xs), max(ys)


def describe_device():
    if DEVICE is not None:
        return f"device forced to {DEVICE}"
    try:
        import torch
        if torch.cuda.is_available():
            return "GPU: " + torch.cuda.get_device_name(0)
        return "CPU (no CUDA GPU found)"
    except Exception:
        return "device unknown"

## Part 5 — The detection pipeline

For every frame:

1. **YOLO** (with tracking) returns the people boxes and a stable ID per person.
2. For each person, the **head area** (top `HEAD_FRAC` of the box) is cropped, enlarged if small, and given to the Face Landmarker. Only `FACES_PER_FRAME` people are scanned per frame (the ones scanned longest ago first).
3. The landmarks are stored **relative to the head area**, so between two scans they are re-projected onto the person's current box. The face follows the person smoothly without scanning every frame.

`warmup()` runs both models once on blank images so that the first real frames are not slow. If tracking is unavailable, the pipeline falls back to plain detection (a face scan on every person each frame).

In [40]:
class PersonFacePipeline:

    def __init__(self):
        self.person = self._load_yolo()
        self.face = vision.FaceLandmarker.create_from_options(
            vision.FaceLandmarkerOptions(
                base_options=mp_python.BaseOptions(model_asset_path=FACE_MODEL_PATH),
                running_mode=vision.RunningMode.IMAGE,
                num_faces=1,                              # one face per head crop
                min_face_detection_confidence=FACE_CONF,
                min_face_presence_confidence=FACE_CONF,
            ))
        self.show_boxes = True                            # key b
        self.show_mesh = False                            # key m
        self.show_dots = False                            # key l
        self.use_track = True
        self.states = {}                                  # per person: face landmarks, last scan...
        self.frame_i = 0

    # ---- internals -------------------------------------------------------
    @staticmethod
    def _load_yolo():
        try:
            return YOLO(MODEL_NAME)
        except Exception as e:
            print(f"Could not load {MODEL_NAME} ({e}).")
            print(f"Falling back to {MODEL_FALLBACK}. (pip install -U ultralytics may fix this.)")
            return YOLO(MODEL_FALLBACK)

    def _run_yolo(self, frame):
        kw = dict(classes=[0], conf=PERSON_CONF, imgsz=YOLO_IMGSZ, verbose=False)   # class 0 = person
        if DEVICE is not None:
            kw["device"] = DEVICE
        if self.use_track:
            try:
                return self.person.track(frame, persist=True, tracker=TRACKER, **kw)[0]
            except Exception as e:
                print("Tracking unavailable, using plain detection:", e)
                self.use_track = False
        return self.person.predict(frame, **kw)[0]

    @staticmethod
    def _head_region(box, w, h):
        x1, y1, x2, y2 = box
        bw, bh = x2 - x1, y2 - y1
        rx0, rx1 = max(0, int(x1 - 0.1 * bw)), min(w, int(x2 + 0.1 * bw))
        ry0, ry1 = max(0, int(y1 - 0.05 * bh)), min(h, int(y1 + HEAD_FRAC * bh))
        return rx0, ry0, rx1, ry1

    def _scan_face(self, rgb, p, w, h):
        st = p["state"]
        st["last_scan"] = self.frame_i
        rx0, ry0, rx1, ry1 = self._head_region(p["box"], w, h)
        cw, ch = rx1 - rx0, ry1 - ry0
        if cw < 16 or ch < 16:
            return
        crop = rgb[ry0:ry1, rx0:rx1]
        s = FACE_MIN_SIDE / max(cw, ch)
        if s > 1:                                         # enlarge small (far) heads
            crop = cv2.resize(crop, None, fx=s, fy=s, interpolation=cv2.INTER_CUBIC)
        crop = np.ascontiguousarray(crop)
        res = self.face.detect(mp.Image(image_format=mp.ImageFormat.SRGB, data=crop))
        if res.face_landmarks:
            st["rel"] = [(q.x, q.y) for q in res.face_landmarks[0]]   # relative to the head area
            st["miss"] = 0
        else:
            st["miss"] += 1
            if st["miss"] >= 3:                           # face lost for 3 scans -> forget it
                st["rel"] = None

    # ---- public ----------------------------------------------------------
    def detect(self, frame):
        """frame: full-resolution BGR image -> list of people (dicts with box, id, conf, face)."""
        h, w = frame.shape[:2]
        self.frame_i += 1
        if not self.use_track:
            self.states = {}                              # no IDs -> nothing to remember

        # 1) persons
        res = self._run_yolo(frame)
        people = []
        boxes = res.boxes
        if boxes is not None and len(boxes):
            xyxy = boxes.xyxy.cpu().numpy()
            confs = boxes.conf.cpu().numpy()
            if self.use_track and boxes.id is not None:
                ids = boxes.id.int().cpu().tolist()
            else:
                ids = [None] * len(xyxy)
            for i, (b, c, tid) in enumerate(zip(xyxy, confs, ids)):
                key = tid if tid is not None else f"n{i}"
                st = self.states.setdefault(key, {"rel": None, "last_scan": -10**9,
                                                  "miss": 0, "seen": 0})
                st["seen"] = self.frame_i
                people.append({"key": key, "id": tid, "box": tuple(float(v) for v in b),
                               "conf": float(c), "state": st, "face": None})

        # 2) face scans: the people not scanned for the longest time first
        due = [p for p in people if self.frame_i - p["state"]["last_scan"] >= FACE_EVERY]
        due.sort(key=lambda p: p["state"]["last_scan"])
        if due:
            rgb = np.ascontiguousarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            for p in due[:FACES_PER_FRAME]:
                self._scan_face(rgb, p, w, h)

        # 3) re-project the stored landmarks onto each person's current head area
        for p in people:
            rel = p["state"]["rel"]
            if rel:
                rx0, ry0, rx1, ry1 = self._head_region(p["box"], w, h)
                cw, ch = rx1 - rx0, ry1 - ry0
                p["face"] = [(int(rx0 + u * cw), int(ry0 + v * ch)) for u, v in rel]

        # forget people not seen for a while
        self.states = {k: v for k, v in self.states.items() if self.frame_i - v["seen"] <= 30}
        return people

    def warmup(self, w, h):
        """Run both models once on blank images so the first real frames are not slow."""
        dummy = np.zeros((h, w, 3), np.uint8)
        for _ in range(2):
            self.detect(dummy)
        blank_head = np.zeros((FACE_MIN_SIDE, FACE_MIN_SIDE, 3), np.uint8)
        self.face.detect(mp.Image(image_format=mp.ImageFormat.SRGB, data=blank_head))
        self.states = {}
        self.frame_i = 0

    def draw(self, view, people, scale):
        # Mouth / eyes / irises / eyebrows: points to hide (computed once, then cached)
        if not hasattr(self, "_hide_idx"):
            groups = (Conn.FACE_LANDMARKS_LIPS,
                      Conn.FACE_LANDMARKS_LEFT_EYE, Conn.FACE_LANDMARKS_RIGHT_EYE,
                      Conn.FACE_LANDMARKS_LEFT_IRIS, Conn.FACE_LANDMARKS_RIGHT_IRIS,
                      Conn.FACE_LANDMARKS_LEFT_EYEBROW, Conn.FACE_LANDMARKS_RIGHT_EYEBROW)
            hide_pairs = {(c.start, c.end) for g in groups for c in g}
            self._hide_idx = {i for pair in hide_pairs for i in pair}
        RIGHT_IRIS, LEFT_IRIS = 468, 473          # iris centers = the two red dots

        n_faces = 0
        for p in people:
            if self.show_boxes:
                x1, y1, x2, y2 = [int(v * scale) for v in p["box"]]
                cv2.rectangle(view, (x1, y1), (x2, y2), (255, 160, 0), 2)
                label = f"#{p['id']}" if p["id"] is not None else "person"
                cv2.putText(view, f"{label} {p['conf']:.2f}", (x1, max(y1 - 6, 12)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 160, 0), 1, cv2.LINE_AA)
            if p["face"]:
                n_faces += 1
                pts = [(int(x * scale), int(y * scale)) for x, y in p["face"]]
                if self.show_mesh:
                    draw_connections(view, pts, Conn.FACE_LANDMARKS_TESSELATION, (90, 90, 90))

                # eyes = two red dots
                if len(pts) > LEFT_IRIS:
                    for idx in (RIGHT_IRIS, LEFT_IRIS):
                        cv2.circle(view, pts[idx], 3, (0, 0, 255), -1, cv2.LINE_AA)

                # optional landmark dots (key l), without mouth, eyes and eyebrows
                if self.show_dots:
                    for i, (x, y) in enumerate(pts):
                        if i not in self._hide_idx and i not in (RIGHT_IRIS, LEFT_IRIS):
                            cv2.circle(view, (x, y), 1, (0, 255, 255), -1)

                # green box around the face
                fx1, fy1, fx2, fy2 = bbox(pts)
                cv2.rectangle(view, (fx1, fy1), (fx2, fy2), (0, 255, 0), 2)
                cv2.putText(view, f"people: {len(people)}  faces: {n_faces}", (10, 50),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)       
                
    def on_key(self, key):
        if key == ord("b"):
            self.show_boxes = not self.show_boxes
        elif key == ord("m"):
            self.show_mesh = not self.show_mesh
        elif key == ord("l"):
            self.show_dots = not self.show_dots

    def close(self):
        self.face.close()

## Part 6 — Camera loop

`show_feed(pipeline)` opens the camera, runs the pipeline on every full-resolution frame, then resizes **only the copy shown on screen**.
This cell only *defines* the function; the camera is **not** opened here.

In [41]:
def window_closed():
    """True once the user clicked the X in the window's title bar."""
    try:
        visible = cv2.getWindowProperty(WINDOW, cv2.WND_PROP_VISIBLE)
        autosize = cv2.getWindowProperty(WINDOW, cv2.WND_PROP_AUTOSIZE)
        return visible < 1 or autosize < 0
    except cv2.error:
        return True


def show_feed(pipeline):
    cap = cv2.VideoCapture(SOURCE)
    cap.set(cv2.CAP_PROP_FOURCC, cv2.VideoWriter_fourcc(*"MJPG"))   # lets webcams deliver HD+ at good FPS
    cap.set(cv2.CAP_PROP_FRAME_WIDTH, CAM_W)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, CAM_H)
    cap.set(cv2.CAP_PROP_BUFFERSIZE, 1)                              # always the freshest frame

    if not cap.isOpened():
        print("Cannot open the camera. Try another SOURCE (1, 2...) or close apps using it.")
        return

    real_w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    real_h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    print(f"Camera opened at {real_w}x{real_h} (requested {CAM_W}x{CAM_H}).")
    for _ in range(5):                                               # let auto-exposure settle
        cap.read()

    n, fps, t_prev = 0, 0.0, time.perf_counter()
    try:
        while True:
            ok, frame = cap.read()
            if not ok:
                print("No frame received (camera lost or video ended).")
                break
            n += 1

            people = pipeline.detect(frame)                          # full resolution

            h, w = frame.shape[:2]
            scale = WIDTH / w
            view = cv2.resize(frame, (WIDTH, int(h * scale)))        # display copy only
            pipeline.draw(view, people, scale)

            t_now = time.perf_counter()
            fps = 0.9 * fps + 0.1 / max(t_now - t_prev, 1e-6) if n > 1 else 0.0
            t_prev = t_now
            cv2.putText(view, f"{fps:.0f} fps", (10, 25),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)

            cv2.imshow(WINDOW, view)
            key = cv2.waitKey(1) & 0xFF
            if key == ord("q"):
                break
            if key != 255:
                pipeline.on_key(key)
            if n > 2 and window_closed():                            # checked BEFORE the next imshow
                break
    except KeyboardInterrupt:
        print("Stopped.")
    finally:
        cap.release()
        cv2.destroyAllWindows()
        cv2.waitKey(1)
        print(f"Camera released. {n} frames shown.")

## Part 7 — Run

Order of events, so the camera opens **once** and only when everything is ready:

1. Load YOLO and the face model (the first run downloads the YOLO weights).
2. Warm both models up on blank images (including the tracker).
3. **Only then** open the camera and show the live feed.
4. When you quit, release the camera and the models.

You can re-run this cell as many times as you like.

In [42]:
print("Loading models...")
pipeline = PersonFacePipeline()

print("Warming up...")
pipeline.warmup(CAM_W, CAM_H)

print(f"Everything is loaded and ready ({describe_device()}). Opening the camera now.")
try:
    show_feed(pipeline)          # q or X = quit, b = boxes, m = mesh, l = dots
finally:
    pipeline.close()

Loading models...
Warming up...
Everything is loaded and ready (CPU (no CUDA GPU found)). Opening the camera now.
Camera opened at 1280x720 (requested 1920x1080).
Camera released. 62 frames shown.


## Tuning guide

| Problem | Change (Part 2) |
|---------|-----------------|
| People still missed far away | bigger model (`yolo26m.pt`), higher `YOLO_IMGSZ` (1536 / 1920), `CAM_W, CAM_H = 3840, 2160`, lower `PERSON_CONF` (0.15) |
| FPS too low | smaller model (`yolo26n.pt`), lower `YOLO_IMGSZ` (960 / 640), lower `FACES_PER_FRAME`, raise `FACE_EVERY` |
| Person found but no face | the face is probably too small or turned away; enlarge the head area with `HEAD_FRAC = 0.5`, or lower `FACE_CONF` |
| Ghost people | raise `PERSON_CONF` (0.35–0.5) |
| Slow even with a good PC | `describe_device()` says CPU: install a CUDA build of PyTorch to use your NVIDIA GPU |
| `Camera opened at` shows a lower resolution than requested | the camera does not support it; the log shows the real value |

YOLO only finds *bodies*; a face still needs roughly 20–25 px of width to get reliable landmarks, so the face limit is lower than the person limit.